# Appending Zarr v2 slices with time-series-friendly chunks

This notebook generates and appends 31 daily Zarr v2 slices in `inputs/zarr-v2`. Zarr v2 is the default target format, so `zarr_version` is deliberately omitted. The output is written below `outputs`, which is ignored by Git.

In [2]:
import os
import shutil
from pathlib import Path

import xarray as xr
from create_inputs import make_zarr_v2_slices

from zappend.api import zappend

## Generate daily input slices

The helper creates deterministic daily slices with dimensions `(time=1, lat=1800, lon=3600)`. Each day has a distinct constant value, which makes it easy to check that all inputs were appended in order.

In [2]:
%%time
make_zarr_v2_slices(31)

CPU times: user 5.16 s, sys: 880 ms, total: 6.04 s
Wall time: 4.15 s


## Collect the inputs

`zappend` accepts paths directly. Sorting the paths puts the daily slices in chronological order. The same list is used for both target layouts below.

In [3]:
input_dir = Path("inputs/zarr-v2")
output_dir =Path("outputs")
output_dir.mkdir(exist_ok=True)
slices = sorted(str(path) for path in input_dir.glob('*.zarr'))
slices

['inputs/zarr-v2/slice-2024-01-01.zarr',
 'inputs/zarr-v2/slice-2024-01-02.zarr',
 'inputs/zarr-v2/slice-2024-01-03.zarr',
 'inputs/zarr-v2/slice-2024-01-04.zarr',
 'inputs/zarr-v2/slice-2024-01-05.zarr',
 'inputs/zarr-v2/slice-2024-01-06.zarr',
 'inputs/zarr-v2/slice-2024-01-07.zarr',
 'inputs/zarr-v2/slice-2024-01-08.zarr',
 'inputs/zarr-v2/slice-2024-01-09.zarr',
 'inputs/zarr-v2/slice-2024-01-10.zarr',
 'inputs/zarr-v2/slice-2024-01-11.zarr',
 'inputs/zarr-v2/slice-2024-01-12.zarr',
 'inputs/zarr-v2/slice-2024-01-13.zarr',
 'inputs/zarr-v2/slice-2024-01-14.zarr',
 'inputs/zarr-v2/slice-2024-01-15.zarr',
 'inputs/zarr-v2/slice-2024-01-16.zarr',
 'inputs/zarr-v2/slice-2024-01-17.zarr',
 'inputs/zarr-v2/slice-2024-01-18.zarr',
 'inputs/zarr-v2/slice-2024-01-19.zarr',
 'inputs/zarr-v2/slice-2024-01-20.zarr',
 'inputs/zarr-v2/slice-2024-01-21.zarr',
 'inputs/zarr-v2/slice-2024-01-22.zarr',
 'inputs/zarr-v2/slice-2024-01-23.zarr',
 'inputs/zarr-v2/slice-2024-01-24.zarr',
 'inputs/zarr-v2

## Default layout: favor spatial access

This first call keeps the input encoding. The source data use one day per time chunk and 450 by 450 spatial chunks, a sensible layout when maps or single days are the common access pattern. `force_new=True` makes the notebook repeatable by replacing only this example target.

In [4]:
%%time
full_time_chunk = len(slices)
target = f"{output_dir}/v2_example_space_opt.zarr"
count = zappend(
    slices,
    target_dir=target,
    force_new=True,
    logging="INFO",
)
assert count == full_time_chunk

2026-10-29 16:20:01,418 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-10-29 16:20:01,421 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-01.zarr
2026-10-29 16:20:01,431 INFO Creating target dataset outputs/v2_example_space_opt.zarr
2026-10-29 16:20:01,561 INFO Transaction completed.
2026-10-29 16:20:01,562 INFO Slice dataset inputs/zarr-v2/slice-2024-01-01.zarr closed
2026-10-29 16:20:01,572 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-02.zarr
2026-10-29 16:20:01,580 INFO Updating target dataset outputs/v2_example_space_opt.zarr
2026-10-29 16:20:01,722 INFO Transaction completed.
2026-10-29 16:20:01,723 INFO Slice dataset inputs/zarr-v2/slice-2024-01-02.zarr closed
2026-10-29 16:20:01,730 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-03.zarr
2026-10-29 16:20:01,738 INFO Updating target dataset outputs/v2_example_space_opt.zarr
2026-10-29 16:20:01,882 INFO Transaction completed.
2026-

CPU times: user 6.65 s, sys: 1.07 s, total: 7.72 s
Wall time: 5.64 s


Opening the target shows its dimensions and the chunks inherited from the first slice. Compare this with the second target below.

In [5]:
%%time
ds = xr.open_zarr(target)
ds

CPU times: user 21.2 ms, sys: 3.18 ms, total: 24.4 ms
Wall time: 20 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

## Time-series layout

Here the data variables are written with one 31-day time chunk. A request for a point time series therefore reads one temporal chunk instead of 31 daily chunks. This trades more data read for map requests against fewer reads for time-series requests.

In [6]:
%%time
full_time_chunk = len(slices)
target = f"{output_dir}/v2_example_time_opt.zarr"
count = zappend(
    slices,
    target_dir=target,
    force_new=True,
    variables={
        'temperature': {'encoding': {'chunks': [full_time_chunk, 90, 90]}},
        'quality_flag': {'encoding': {'chunks': [full_time_chunk, 90, 90]}},
    },
    logging="INFO",
)
assert count == full_time_chunk

2026-10-29 16:20:07,169 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-10-29 16:20:07,170 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-01.zarr
2026-10-29 16:20:07,182 INFO Creating target dataset outputs/v2_example_time_opt.zarr
2026-10-29 16:20:08,898 INFO Transaction completed.
2026-10-29 16:20:08,898 INFO Slice dataset inputs/zarr-v2/slice-2024-01-01.zarr closed
2026-10-29 16:20:08,906 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-02.zarr
2026-10-29 16:20:08,913 INFO Updating target dataset outputs/v2_example_time_opt.zarr
2026-10-29 16:20:11,707 INFO Transaction completed.
2026-10-29 16:20:11,707 INFO Slice dataset inputs/zarr-v2/slice-2024-01-02.zarr closed
2026-10-29 16:20:11,716 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-03.zarr
2026-10-29 16:20:11,726 INFO Updating target dataset outputs/v2_example_time_opt.zarr
2026-10-29 16:20:14,530 INFO Transaction completed.
2026-10-

CPU times: user 1min 54s, sys: 27.8 s, total: 2min 22s
Wall time: 1min 29s


In [7]:
%%time
ds = xr.open_zarr(target)
ds

CPU times: user 18.3 ms, sys: 3.11 ms, total: 21.4 ms
Wall time: 17 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

In [4]:
if os.path.exists(input_dir):
    shutil.rmtree(input_dir)